# Prepare Electron Microscopy Data

The powerfit program requires a EM density of a unknown structure where it can fit structures into.

Most of the the EM density contains multiple structures, so we want to remove all but the unknown structure.

In this example we will use the [phenix software](https://www.phenix-online.org/) to prepare the EM density.
Make sure you have phenix installed and its commands like `phenix.about` are callable from the command line.

For this example we will use [EMD-33292](https://www.ebi.ac.uk/emdb/EMD-33292), a sodium channel, with fitted model [7xm9](https://www.ebi.ac.uk/pdbe/entry/pdb/7xm9).

TODO use Mol* to visualize the EM density and fitted model.

Lets start by downloading the density map and the fitted model.

In [1]:
!wget -nc https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-33292/map/emd_33292.map.gz
!gunzip emd_33292.map.gz
!wget -nc https://www.ebi.ac.uk/pdbe/entry-files/download/7xm9.cif

File ‘emd_33292.map.gz’ already there; not retrieving.

File ‘7xm9.cif’ already there; not retrieving.



In [16]:
from pathlib import Path

from molviewspec import MVSJ, ComponentExpression, create_builder, molstar_notebook

# Create builder
builder = create_builder()

# Add the density map with gray translucent volume
# builder.download(url="https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-33292/map/emd_33292.map.gz").parse(format="map").volume().representation(
#     type="isosurface",
#     relative_isovalue=1.0
# ).color(color="gray").opacity(opacity=0.3)

# Add the structure and color chains differently
structure = (
    builder.download(url="https://www.ebi.ac.uk/pdbe/entry-files/download/7xm9.cif")
    .parse(format="mmcif")
    .model_structure()
)

# Chain A - green
structure.component(selector=ComponentExpression(label_asym_id="A")).representation().color(color="green")

# Chain B - orange
structure.component(selector=ComponentExpression(label_asym_id="B")).representation().color(color="orange")

# Chain C - purple
structure.component(selector=ComponentExpression(label_asym_id="C")).representation().color(color="purple")

# Get the state and prepare data
state = builder.get_state()
MVSJ(data=state).dump(Path("emd_33292.7xm9.mvsj"))

In [ ]:
# Render in molstar
molstar_notebook(state=state, width=1200, height=800)

Phenix has a command called [phenix.map_box](https://phenix-online.org/documentation/reference/map_box.html) that can be used to mask out the known model from the EM density.

In [ ]:
!phenix.map_box 7xm9.cif emd_33292.map selection=B